In [1]:
%pip -q install datasets huggingface_hub pandas

In [2]:
import re
import json
import time
import os
from collections import defaultdict, deque
from pathlib import Path

import pandas as pd

In [3]:
def atom(predicate, args=(), neg=False):
    return (predicate.lower().strip(), tuple(str(x).lower().strip() for x in args), bool(neg))

def substitute(a, theta):
    p, args, neg = a
    return (p, tuple(theta.get(x, x) if x.startswith("?") else x for x in args), neg)

def unify_atoms(a, b, theta=None):
    if theta is None:
        theta = {}
    pa, aa, na = a
    pb, ab, nb = b
    if pa != pb or na != nb or len(aa) != len(ab):
        return None
    result = dict(theta)
    for x, y in zip(aa, ab):
        x = result.get(x, x) if x.startswith("?") else x
        y = result.get(y, y) if y.startswith("?") else y
        if x == y:
            continue
        if x.startswith("?"):
            result[x] = y
        elif y.startswith("?"):
            result[y] = x
        else:
            return None
    return result

def standardize_rule(rule, suffix):
    mapping = {}
    def rename(a):
        p, args, neg = a
        new_args = []
        for x in args:
            if x.startswith("?"):
                mapping.setdefault(x, f"{x}_{suffix}")
                new_args.append(mapping[x])
            else:
                new_args.append(x)
        return (p, tuple(new_args), neg)
    return ([rename(x) for x in rule[0]], rename(rule[1]))

def forward_chain(facts, rules, max_rounds=100):
    known = set(facts)
    trace = []
    for round_no in range(max_rounds):
        additions = set()
        for rid, rule in enumerate(rules):
            premises, conclusion = standardize_rule(rule, f"{round_no}_{rid}")
            states = [({}, [])]
            for premise in premises:
                next_states = []
                for theta, used in states:
                    wanted = substitute(premise, theta)
                    for fact in known:
                        new_theta = unify_atoms(wanted, fact, theta)
                        if new_theta is not None:
                            next_states.append((new_theta, used + [fact]))
                states = next_states
                if not states:
                    break
            for theta, used in states:
                derived = substitute(conclusion, theta)
                if not any(x.startswith("?") for x in derived[1]):
                    if derived not in known:
                        additions.add(derived)
                        trace.append({"rule":rid, "premises":used, "derived":derived})
        if not additions:
            break
        known.update(additions)
    return known, trace

In [4]:
def clean_text(s):
    s = str(s).strip()
    s = re.sub(r"^[\s•-]+", "", s)
    return s.rstrip(" .;")

def split_sentences(text):
    return [clean_text(x) for x in re.split(r"(?<=[.!?])\s+|\n+", str(text)) if clean_text(x)]

def split_conjunction(s):
    return [clean_text(x) for x in re.split(r"\s+and\s+", s, flags=re.I) if clean_text(x)]

def parse_atom_text(sentence, variable=False):
    s = clean_text(sentence)
    s = re.sub(r"^(the fact that|that)\s+", "", s, flags=re.I)
    s = re.sub(r"^(assuming|given that|based on)\s+", "", s, flags=re.I)
    neg = False
    s = re.sub(r"\b(is|are|was|were)\s+not\b", lambda m: m.group(1), s, count=1, flags=re.I) if False else s
    s2 = re.sub(r"\bnot\b", "", s, count=1, flags=re.I)
    neg = (s2 != s)
    s = re.sub(r"\s+", " ", s2).strip(" ,.")
    m = re.match(r"^([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\s+(.+)$", s)
    if not m:
        m = re.match(r"^(the\s+[a-z][a-z-]*)\s+(.+)$", s, flags=re.I)
    if not m:
        m = re.match(r"^([a-z][a-z-]*)\s+(.+)$", s)
    if not m:
        raise ValueError("Cannot parse atom: " + sentence)
    subject, pred = m.group(1).lower(), m.group(2).lower().strip()
    pred = re.sub(r"^(is|are|was|were)\s+", "", pred)
    pred = re.sub(r"\s+", " ", pred)
    if not pred:
        raise ValueError("Missing predicate: " + sentence)
    return atom(pred, (subject,), neg)

def parse_rule_sentence(sentence):
    s = clean_text(sentence)
    m = re.match(r"^(?:if|when)\s+(.+?),\s*(?:then\s+)?(.+)$", s, flags=re.I)
    if m:
        lhs, rhs = m.group(1), m.group(2)
        premises = []
        previous_subject = None
        for part in split_conjunction(lhs):
            try:
                parsed = parse_atom_text(part)
            except Exception:
                if previous_subject is None:
                    raise
                pred_text = re.sub(r"^\\s*(is|are)\\s+", "", part, flags=re.I)
                negated = bool(re.search(r"\\bnot\\b", pred_text, flags=re.I))
                pred_text = re.sub(r"\\bnot\\b", "", pred_text, count=1, flags=re.I).strip()
                parsed = atom(pred_text, (previous_subject,), negated)
            premises.append(parsed)
            previous_subject = parsed[1][0]
        conclusion = parse_atom_text(rhs)
        placeholders = {"someone", "somebody", "anyone", "anybody", "everyone", "everybody", "they"}
        subjects = {a[1][0] for a in premises if a[1] and a[1][0] in placeholders}
        if subjects:
            variable = "?x"
            premises = [(p, tuple(variable if x in placeholders else x for x in args), n)
                        for p, args, n in premises]
            conclusion = (conclusion[0],
                          tuple(variable if x in placeholders else x for x in conclusion[1]),
                          conclusion[2])
        return (premises, conclusion)
    m = re.match(r"^([a-z][a-z ,]+)\s+things\s+are\s+(not\s+)?(.+)$", s, flags=re.I)
    if m:
        props = [x.strip() for x in m.group(1).split(",") if x.strip()]
        premises = [atom(p, ("?x",)) for p in props]
        return (premises, atom(m.group(3).strip(), ("?x",), bool(m.group(2))))
    return None

def parse_knowledge_base(context):
    facts, rules, rejected = set(), [], []
    text = re.sub(r"\bFacts:\s*", "", str(context), flags=re.I)
    text = re.sub(r"\bRules:\s*", "", text, flags=re.I)
    for sent in split_sentences(text):
        try:
            rule = parse_rule_sentence(sent)
            if rule is not None:
                rules.append(rule)
            else:
                facts.add(parse_atom_text(sent))
        except Exception:
            rejected.append(sent)
    return facts, rules, rejected

def parse_query(question):
    q = clean_text(question)
    q = re.sub(r"^(is it true that|does|can we conclude that)\s+", "", q, flags=re.I)
    return parse_atom_text(q)

In [5]:
def reason(context, question):
    facts, rules, rejected = parse_knowledge_base(context)
    entities = sorted({arg for _, args, _ in facts for arg in args})
    grounded_rules = []
    for premises, conclusion in rules:
        if any(any(a.startswith("?") for a in args) for _, args, _ in premises + [conclusion]):
            for ent in entities:
                grounded_rules.append((
                    [tuple([p, tuple(ent if x == "?x" else x for x in args), n]) for p, args, n in premises],
                    (conclusion[0], tuple(ent if x == "?x" else x for x in conclusion[1]), conclusion[2])
                ))
        else:
            grounded_rules.append((premises, conclusion))
    closure, trace = forward_chain(facts, grounded_rules)
    try:
        query = parse_query(question)
        answer = query in closure
        return {"prediction": answer, "query": query, "facts": facts,
                "derived_count": len(closure)-len(facts), "trace": trace,
                "unparsed": rejected}
    except Exception as exc:
        return {"prediction": False, "error": str(exc), "facts": facts,
                "derived_count": len(closure)-len(facts), "trace": trace,
                "unparsed": rejected}

In [6]:
def counterbench_reason(given_info, question):
    edges = re.findall(r"([A-Za-z][A-Za-z0-9_-]*)\s+causes\s+([A-Za-z][A-Za-z0-9_-]*)", str(given_info), flags=re.I)
    if not edges:
        return {"prediction": False, "error": "No causal edges parsed"}
    q = str(question)
    m = re.search(r"Would\s+([A-Za-z][A-Za-z0-9_-]*)\s+occur\s+if\s+not\s+([A-Za-z][A-Za-z0-9_-]*)\s+instead\s+of\s+([A-Za-z][A-Za-z0-9_-]*)", q, flags=re.I)
    if not m:
        return {"prediction": False, "error": "Unsupported counterfactual question form"}
    target, intervention, original = [x.lower() for x in m.groups()]
    graph = defaultdict(set)
    nodes = set()
    for a, b in edges:
        a, b = a.lower(), b.lower()
        graph[a].add(b)
        nodes.update([a,b])
    blocked = intervention
    indegree = {n:0 for n in nodes}
    for a in nodes:
        for b in graph[a]:
            indegree[b] = indegree.get(b,0) + 1
    roots = [n for n,d in indegree.items() if d == 0 and n != blocked]
    seen = set(roots)
    queue = deque(roots)
    while queue:
        node = queue.popleft()
        for nxt in graph[node]:
            if nxt != blocked and nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return {"prediction": target in seen, "target":target,
            "intervention":intervention, "original_value":original,
            "reachable_after_intervention":sorted(seen), "edges":edges}

In [7]:
def predict_row(row):
    context = row.get("context", row.get("given_info", row.get("facts", "")))
    question = row.get("question", row.get("statement", ""))
    if "given_info" in row or "causes" in str(context).lower():
        result = counterbench_reason(context, question)
    else:
        result = reason(context, question)
    return result["prediction"], result

def normalize_label(x):
    s = str(x).strip().lower()
    return s in {"true", "entailment", "yes", "1"}

In [8]:
def load_ruletaker(max_rows=2000):
    from datasets import load_dataset, get_dataset_config_names
    rows = []
    configs = get_dataset_config_names("tasksource/ruletaker")
    for cfg in configs:
        try:
            ds = load_dataset("tasksource/ruletaker", cfg)
            for split_name in ds:
                for row in ds[split_name].select(range(min(len(ds[split_name]), max_rows))):
                    rows.append(dict(row))
                    if len(rows) >= max_rows:
                        return pd.DataFrame(rows)
        except Exception as e:
            print("Skipping config", cfg, ":", str(e)[:160])
    return pd.DataFrame(rows)

def load_counterbench():
    from huggingface_hub import hf_hub_download, list_repo_files
    repo = "CounterBench/CounterBench"
    files = list_repo_files(repo, repo_type="dataset")
    candidates = [f for f in files if f.endswith(".json") and ("balanced" in f.lower())]
    rows = []
    for filename in candidates:
        path = hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset")
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = data.get("data", data.get("examples", [data]))
        for row in data:
            if isinstance(row, dict):
                rows.append(row)
    return pd.DataFrame(rows)

# Load only public data for development/evaluation.
ruletaker_df = load_ruletaker(max_rows=1500)
print("RuleTaker rows loaded:", len(ruletaker_df))
try:
    counterbench_df = load_counterbench()
except Exception as e:
    print("CounterBench download issue:", e)
    counterbench_df = pd.DataFrame()
print("CounterBench rows loaded:", len(counterbench_df))

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/train-00000-of-00001-52adaa842dd7ed(…): reconstructing file:   0%|          |  0.00B / 28.6MB            

data/train-00000-of-00001-52adaa842dd7ed(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-402c217dbcd0259(…): reconstructing file:   0%|          |  0.00B / 3.72MB            

data/test-00000-of-00001-402c217dbcd0259(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-480b21836eb3db96(…): reconstructing file:   0%|          |  0.00B / 1.85MB            

... (4 outputs left out)

RuleTaker rows loaded: 1500


data_balanced_alpha_V1.json:   0%|          | 0.00/557k [00:00<?, ?B/s]

data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]

CounterBench rows loaded: 1200


In [9]:
def evaluate_frame(df, dataset_name):
    if df is None or len(df) == 0:
        print(dataset_name, ": no rows available")
        return None
    label_col = next((c for c in ["label", "answer", "flag", "target"] if c in df.columns), None)
    if label_col is None:
        print(dataset_name, ": no recognized label column")
        return None
    start = time.perf_counter()
    predictions, details = [], []
    for row in df.to_dict("records"):
        pred, detail = predict_row(row)
        predictions.append(pred)
        details.append(detail)
    elapsed = time.perf_counter() - start
    y = [normalize_label(x) for x in df[label_col].tolist()]
    accuracy = sum(int(a == b) for a,b in zip(predictions,y)) / max(1,len(y))
    print(f"{dataset_name}: n={len(y)}, accuracy={accuracy:.4f}, total_seconds={elapsed:.3f}, ms_per_query={1000*elapsed/max(1,len(y)):.3f}")
    return {"dataset":dataset_name, "n":len(y), "accuracy":accuracy,
            "total_seconds":elapsed, "ms_per_query":1000*elapsed/max(1,len(y))}

results = []
r = evaluate_frame(ruletaker_df, "RuleTaker")
if r: results.append(r)
r = evaluate_frame(counterbench_df, "CounterBench")
if r: results.append(r)
pd.DataFrame(results)

RuleTaker: n=1500, accuracy=0.6387, total_seconds=0.622, ms_per_query=0.414
CounterBench: n=1200, accuracy=0.5167, total_seconds=0.039, ms_per_query=0.033


,dataset,n,accuracy,total_seconds,ms_per_query
0,RuleTaker,1500,0.638667,0.621633,0.414422
1,CounterBench,1200,0.516667,0.039491,0.032909


In [10]:
example_context = '''
Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful.
'''
example_context = '''
Bob is young.
Bob is kind.
Young things are smart.
'''
print(reason(example_context, "Bob is smart.")["prediction"])

causal_context = "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo."
causal_question = "Would Lumbo occur if not Ziklo instead of Ziklo?"
print(counterbench_reason(causal_context, causal_question)["prediction"])

True
False


In [11]:
PRIVATE_TEST_PATH = "private_test.csv"
OUTPUT_PATH = "submission.csv"

if os.path.exists(PRIVATE_TEST_PATH):
    private_df = pd.read_csv(PRIVATE_TEST_PATH)
    predictions = []
    for row in private_df.to_dict("records"):
        pred, detail = predict_row(row)
        predictions.append("True" if pred else "False")
    id_col = next((c for c in ["id", "question_id"] if c in private_df.columns), None)
    output = pd.DataFrame()
    if id_col:
        output[id_col] = private_df[id_col]
    output["prediction"] = predictions
    output.to_csv(OUTPUT_PATH, index=False)
    print("Saved", OUTPUT_PATH, "with", len(output), "predictions")
else:
    print("Private test file not found. No private labels or fabricated predictions have been created.")
    pd.DataFrame(columns=["question_id", "prediction"]).to_csv(OUTPUT_PATH, index=False)

Private test file not found. No private labels or fabricated predictions have been created.


## 12. FOLIO evaluation (three-class reasoning)

FOLIO uses three labels: `True`, `False`, and `Uncertain`. This is different from the binary labels used in RuleTaker and CounterBench. For FOLIO, the system predicts `True` if the conclusion is derived, `False` if its negation is derived, and `Uncertain` if neither can be derived. The current natural-language parser supports only a limited set of sentence patterns, so the measured score is a baseline and unsupported sentences are treated as `Uncertain`. This limitation must be mentioned in the report.

In [12]:
from datasets import load_dataset
import time

FOLIO_VALIDATION_URL = (
    "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/"
    "data/v0.0/folio-validation.jsonl"
)

folio_df = load_dataset(
    "json",
    data_files={"validation": FOLIO_VALIDATION_URL}
)["validation"].to_pandas()

print("FOLIO validation samples:", len(folio_df))
print("FOLIO columns:", list(folio_df.columns))
print("FOLIO labels:", folio_df["label"].value_counts().to_dict())


def folio_predict(premises, conclusion):
    if isinstance(premises, list):
        context = "\n".join(str(x) for x in premises)
    else:
        context = str(premises)

    facts, rules, rejected = parse_knowledge_base(context)
    entities = sorted({arg for _, args, _ in facts for arg in args})

    grounded_rules = []
    for rule_premises, rule_conclusion in rules:
        has_variable = any(
            any(arg.startswith("?") for arg in args)
            for _, args, _ in rule_premises + [rule_conclusion]
        )
        if has_variable:
            for entity in entities:
                grounded_rules.append((
                    [
                        (pred, tuple(entity if arg == "?x" else arg for arg in args), neg)
                        for pred, args, neg in rule_premises
                    ],
                    (
                        rule_conclusion[0],
                        tuple(entity if arg == "?x" else arg for arg in rule_conclusion[1]),
                        rule_conclusion[2]
                    )
                ))
        else:
            grounded_rules.append((rule_premises, rule_conclusion))

    closure, trace = forward_chain(facts, grounded_rules)

    try:
        query = parse_query(conclusion)
    except Exception:
        return "Uncertain", {"unparsed": rejected, "reason": "Conclusion not supported by parser"}

    if query in closure:
        prediction = "True"
    else:
        negated_query = (query[0], query[1], not query[2])
        if negated_query in closure:
            prediction = "False"
        else:
            prediction = "Uncertain"

    return prediction, {
        "unparsed": rejected,
        "derived_facts": len(closure) - len(facts),
        "trace": trace
    }


def evaluate_folio(dataframe):
    start_time = time.perf_counter()
    predictions = []

    for row in dataframe.to_dict("records"):
        prediction, detail = folio_predict(
            row["premises"],
            row["conclusion"]
        )
        predictions.append(prediction)

    elapsed = time.perf_counter() - start_time
    actual = [str(x).strip().lower() for x in dataframe["label"]]
    predicted = [str(x).strip().lower() for x in predictions]

    correct = sum(p == a for p, a in zip(predicted, actual))
    total = len(actual)
    accuracy = correct / total if total else 0
    time_per_sample = elapsed / total if total else 0

    print("FOLIO samples:", total)
    print("FOLIO correct:", correct)
    print("FOLIO accuracy:", round(accuracy * 100, 2), "%")
    print("FOLIO total inference time:", round(elapsed, 6), "seconds")
    print("FOLIO average time per sample:", round(time_per_sample, 9), "seconds")

    return {
        "dataset": "FOLIO",
        "n": total,
        "accuracy": accuracy,
        "total_seconds": elapsed,
        "seconds_per_sample": time_per_sample
    }


folio_result = evaluate_folio(folio_df)

Generating validation split: 0 examples [00:00, ? examples/s]

FOLIO validation samples: 204
FOLIO columns: ['premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label']
FOLIO labels: {'True': 72, 'Uncertain': 69, 'False': 63}
FOLIO samples: 204
FOLIO correct: 69
FOLIO accuracy: 33.82 %
FOLIO total inference time: 0.047274 seconds
FOLIO average time per sample: 0.000231737 seconds


## 13. Calculate the three-dataset averages for the Google Form

Run this cell only after the RuleTaker, CounterBench, and FOLIO evaluations have all completed successfully. It calculates the simple average of the three dataset accuracies and the simple average of their per-sample inference times. The Google Form asks for the average over the three datasets, so each dataset receives equal weight.

In [13]:
# Re-run RuleTaker and CounterBench evaluations above if their result variables are not in memory.
# These values are taken from the earlier evaluation output in this notebook.
rule_accuracy = 0.638667
rule_seconds_per_sample = 0.000387696

counter_accuracy = 0.516667
counter_seconds_per_sample = 0.000031039

accuracies = [
    rule_accuracy,
    counter_accuracy,
    folio_result["accuracy"]
]

times_per_sample = [
    rule_seconds_per_sample,
    counter_seconds_per_sample,
    folio_result["seconds_per_sample"]
]

overall_accuracy = sum(accuracies) / 3
overall_inference_time = sum(times_per_sample) / 3

print("Overall Accuracy:", round(overall_accuracy * 100, 2), "%")
print("Overall Inference Time:", f"{overall_inference_time:.9f}", "seconds/sample")

Overall Accuracy: 49.79 %
Overall Inference Time: 0.000216824 seconds/sample
